# Download Under 5 White Non-Hispanic Demographics

Downloads white non-Hispanic population under 5 years old from the Census API (ACS 5-year 2023)
and calculates the percentage at county and state levels.

**Key Census Tables:**
- B01001: Sex by Age (Total Population)
- B01001H: Sex by Age (White Alone, Not Hispanic or Latino)

**Requirements:**
```bash
pip install requests python-dotenv
```

**Setup:**
Create a `.env` file in the project root with:
```
census_api_key=YOUR_API_KEY_HERE
```
Get a free key at: https://api.census.gov/data/key_signup.html


In [ ]:
import os
import json
import requests
from pathlib import Path
from dotenv import load_dotenv

# Load API key from .env file
load_dotenv()
census_api_key = os.getenv("census_api_key")

if not census_api_key:
    raise ValueError("Missing census_api_key in .env file!")

# Output directory
PUBLIC_DIR = (Path.cwd().parent / "public").resolve()
print(f"Output directory: {PUBLIC_DIR}")

# Census API base URL
API_URL = "https://api.census.gov/data/2023/acs/acs5"


In [ ]:
# ============================================================
# STEP 1: Explore available variables in B01001H table
# ============================================================
print("Exploring B01001H (White Alone, Not Hispanic) variables...")

# Get variable metadata
variables_url = f"{API_URL}/variables.json"
response = requests.get(variables_url)
all_variables = response.json()["variables"]

# Filter for B01001H (White Alone Not Hispanic by Age)
b01001h_vars = {k: v for k, v in all_variables.items() if k.startswith("B01001H_") and k.endswith("E")}
print(f"\nFound {len(b01001h_vars)} B01001H estimate variables:\n")

# Show first 25 variables to see the structure
for i, (var_code, var_info) in enumerate(sorted(b01001h_vars.items())[:25]):
    label = var_info.get("label", "No label")
    print(f"  {var_code}: {label}")


In [ ]:
# ============================================================
# STEP 2: Also check B01001 for total population by age
# ============================================================
print("Exploring B01001 (Total Population by Age) variables...")

# Filter for B01001 (Total by Age)
b01001_vars = {k: v for k, v in all_variables.items() if k.startswith("B01001_") and k.endswith("E") and not k.startswith("B01001A")}
print(f"\nFound {len(b01001_vars)} B01001 estimate variables:\n")

# Show first 30 variables
for i, (var_code, var_info) in enumerate(sorted(b01001_vars.items())[:30]):
    label = var_info.get("label", "No label")
    print(f"  {var_code}: {label}")


In [ ]:
# ============================================================
# STEP 3: Identify the specific variables we need
# ============================================================

# From the Census Bureau documentation:
# B01001 - Sex by Age (Total Population)
#   B01001_003E: Male: Under 5 years
#   B01001_027E: Female: Under 5 years
#
# B01001H - Sex by Age (White Alone, Not Hispanic or Latino)
#   B01001H_003E: Male: Under 5 years  
#   B01001H_018E: Female: Under 5 years

# Verify these variables exist and print their labels
key_vars = [
    "B01001_001E",  # Total population
    "B01001_003E",  # Male: Under 5 years (total)
    "B01001_027E",  # Female: Under 5 years (total)
    "B01001H_001E", # White non-Hispanic total
    "B01001H_003E", # White non-Hispanic Male: Under 5 years
    "B01001H_018E", # White non-Hispanic Female: Under 5 years
]

print("Key variables for Under 5 White Non-Hispanic percentage:\n")
for var in key_vars:
    if var in all_variables:
        label = all_variables[var].get("label", "No label")
        print(f"  ✅ {var}: {label}")
    else:
        print(f"  ❌ {var}: NOT FOUND")


In [ ]:
# ============================================================
# STEP 4: Define variables to fetch
# ============================================================

# Variables for Under 5 demographics
VARIABLES = {
    # Total population by age
    "B01001_001E": "total_pop",
    "B01001_003E": "total_male_under5",
    "B01001_027E": "total_female_under5",
    # White non-Hispanic population by age
    "B01001H_001E": "white_nh_total",
    "B01001H_003E": "white_nh_male_under5",
    "B01001H_018E": "white_nh_female_under5",
}

print("Variables to fetch:")
for code, name in VARIABLES.items():
    print(f"  {code} -> {name}")


In [ ]:
# ============================================================
# STEP 5: Download US County Data
# ============================================================
print("Downloading US county under-5 demographics (all states)...")

response = requests.get(API_URL, params={
    "get": "NAME," + ",".join(VARIABLES.keys()),
    "for": "county:*",
    "key": census_api_key,
})
response.raise_for_status()
data = response.json()

# Convert to list of dicts
county_data = []
headers = data[0]
for row in data[1:]:
    record = {"GEOID": row[headers.index("state")] + row[headers.index("county")]}
    record["name"] = row[headers.index("NAME")]
    
    for var_code, var_name in VARIABLES.items():
        val = row[headers.index(var_code)]
        record[var_name] = int(val) if val and val != "-666666666" else 0
    
    # Calculate derived fields
    total_under5 = record["total_male_under5"] + record["total_female_under5"]
    white_nh_under5 = record["white_nh_male_under5"] + record["white_nh_female_under5"]
    
    record["total_under5"] = total_under5
    record["white_nh_under5"] = white_nh_under5
    
    # Calculate percentage: white non-hispanic under 5 / total under 5
    if total_under5 > 0:
        record["white_nh_under5_perc"] = round((white_nh_under5 / total_under5) * 100, 2)
    else:
        record["white_nh_under5_perc"] = None
    
    county_data.append(record)

print(f"  Downloaded {len(county_data)} counties")


In [ ]:
# ============================================================
# STEP 6: Save county data to JSON
# ============================================================

# Save to JSON
output_file = PUBLIC_DIR / "US_county_under5_demographics.json"
with open(output_file, "w") as f:
    json.dump(county_data, f)

print(f"✅ Saved {len(county_data)} counties to {output_file.name}")

# Show some sample records
print("\nSample county records:")
for record in county_data[:3]:
    print(json.dumps(record, indent=2))
    print()


In [ ]:
# ============================================================
# STEP 7: Summary Statistics
# ============================================================
print("\n" + "="*60)
print("SUMMARY STATISTICS")
print("="*60)

# Filter out None values for stats
valid_percs = [r["white_nh_under5_perc"] for r in county_data if r["white_nh_under5_perc"] is not None]

if valid_percs:
    print(f"\nCounties with valid data: {len(valid_percs)}")
    print(f"Min white NH under-5 %: {min(valid_percs):.1f}%")
    print(f"Max white NH under-5 %: {max(valid_percs):.1f}%")
    print(f"Mean white NH under-5 %: {sum(valid_percs)/len(valid_percs):.1f}%")
    
    # Find counties with highest and lowest percentages
    sorted_by_perc = sorted([r for r in county_data if r["white_nh_under5_perc"] is not None], 
                            key=lambda x: x["white_nh_under5_perc"])
    
    print("\n--- Counties with LOWEST white NH under-5 % ---")
    for r in sorted_by_perc[:5]:
        print(f"  {r['name']}: {r['white_nh_under5_perc']:.1f}% ({r['white_nh_under5']} of {r['total_under5']} under-5)")
    
    print("\n--- Counties with HIGHEST white NH under-5 % ---")
    for r in sorted_by_perc[-5:]:
        print(f"  {r['name']}: {r['white_nh_under5_perc']:.1f}% ({r['white_nh_under5']} of {r['total_under5']} under-5)")


In [ ]:
# ============================================================
# STEP 8: Download US State Data
# ============================================================
print("Downloading US state under-5 demographics...")

response = requests.get(API_URL, params={
    "get": "NAME," + ",".join(VARIABLES.keys()),
    "for": "state:*",
    "key": census_api_key,
})
response.raise_for_status()
data = response.json()

# Convert to list of dicts
state_data = []
headers = data[0]
for row in data[1:]:
    record = {"GEOID": row[headers.index("state")]}
    record["name"] = row[headers.index("NAME")]
    
    for var_code, var_name in VARIABLES.items():
        val = row[headers.index(var_code)]
        record[var_name] = int(val) if val and val != "-666666666" else 0
    
    # Calculate derived fields
    total_under5 = record["total_male_under5"] + record["total_female_under5"]
    white_nh_under5 = record["white_nh_male_under5"] + record["white_nh_female_under5"]
    
    record["total_under5"] = total_under5
    record["white_nh_under5"] = white_nh_under5
    
    # Calculate percentage
    if total_under5 > 0:
        record["white_nh_under5_perc"] = round((white_nh_under5 / total_under5) * 100, 2)
    else:
        record["white_nh_under5_perc"] = None
    
    state_data.append(record)

# Save to JSON
output_file = PUBLIC_DIR / "US_state_under5_demographics.json"
with open(output_file, "w") as f:
    json.dump(state_data, f)

print(f"✅ Saved {len(state_data)} states to {output_file.name}")

# Show all states sorted by percentage
print("\nStates sorted by white NH under-5 percentage:")
sorted_states = sorted([s for s in state_data if s["white_nh_under5_perc"] is not None],
                       key=lambda x: x["white_nh_under5_perc"])
for s in sorted_states:
    print(f"  {s['name']}: {s['white_nh_under5_perc']:.1f}%")


In [ ]:
# ============================================================
# DONE
# ============================================================
print("\n" + "="*60)
print("✅ ALL DONE!")
print("="*60)
print("\nCreated files:")
print(f"  - US_county_under5_demographics.json")
print(f"  - US_state_under5_demographics.json")
print("\nThese files contain:")
print("  - total_under5: Total population under 5 years")
print("  - white_nh_under5: White non-Hispanic population under 5")
print("  - white_nh_under5_perc: Percentage that is white non-Hispanic")
